In [ ]:
import kagglehub


path = kagglehub.dataset_download("kritanjalijain/amazon-reviews")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

path = "/root/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2"


print(os.listdir(path))

In [ ]:
import pandas as pd
import os

path = "/root/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2"

df = pd.read_csv(os.path.join(path, "train.csv"), header=None, nrows=50000)

df.columns = ["label", "title", "text"]

df['label'] = df['label'] - 1


df['text'] = df['title'] + " " + df['text']

print(df.head())

In [ ]:
df['text'] = df['text'].fillna("")

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['text'].apply(clean_text)

In [ ]:
print(df['label'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

df['label'].value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['clean_text'])

X = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(X, maxlen=max_len)

y = df['label'].values

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),

    Bidirectional(LSTM(32)),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
drive_path = "/content/drive/MyDrive/bilstm_project/"

In [ ]:

model.save(drive_path + "bilstm_amazon_model.h5")


import pickle
with open(drive_path + "tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)


label_map = {0: "Negative", 1: "Positive"}
with open(drive_path + "label_map.pkl", "wb") as f:
    pickle.dump(label_map, f)

In [ ]:
os.listdir(drive_path)

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype("int32")

In [ ]:
from tensorflow.keras.models import load_model
import pickle

model = load_model(drive_path + "bilstm_amazon_model.h5")

with open(drive_path + "tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open(drive_path + "label_map.pkl", "rb") as f:
    label_map = pickle.load(f)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)
import matplotlib.pyplot as plt

plt.figure()
plt.imshow(cm)

plt.title("Confusion Matrix")
plt.colorbar()

plt.xticks([0, 1], ["Negative", "Positive"])
plt.yticks([0, 1], ["Negative", "Positive"])

plt.xlabel("Predicted")
plt.ylabel("Actual")


for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_sentiment(text):

    text = clean_text(text)


    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=100)


    pred = model.predict(padded)[0][0]

    label = 1 if pred > 0.5 else 0

    return label_map[label]

In [ ]:
user_input = input("Enter your review: ")

result = predict_sentiment(user_input)

print("Predicted Sentiment:", result)